# Visualize physiological and redox data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename

## Load data

In [ ]:
input_filename = "Physiological-Redox-data.csv"
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Physiological-Redox"

df = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[0, 1],
    header=[0, 1],
)

output_filename_fmt = "{variable}.{filetype}"
output_filetype = "pdf"
output_dirpath = REPORTS_DIR / "HK1-R12Ter" / "Physiological-Redox"
output_dirpath.mkdir(parents=True, exist_ok=True)

variable_units_dict = dict(df.columns.to_flat_index())
data = df.droplevel("#", axis=0).droplevel("Unit", axis=1).reset_index(drop=False)
data;

In [ ]:
def plot_physiological_data(data, x, y, ax=None, **kwargs):
    """Plot physiological/redox data as barcharts with individual data points."""
    # Applied to all
    linewidth = kwargs.get("linewidth", 1)
    min_zorder = kwargs.get("min_zorder", 3)
    orient = kwargs.get("orient", "v")
    palette = kwargs.get("palette")

    barplot_kwargs = kwargs.get("barplot_kwargs", {})
    stripplot_kwargs = kwargs.get("stripplot_kwargs", {})
    scatter_kwargs = kwargs.get("scatter_kwargs", {})
    legend_kwargs = kwargs.get("legend_kwargs", {})

    ax = sns.barplot(
        data=data,
        x=x,
        y=y,
        ax=ax,
        hue=x,
        estimator=barplot_kwargs.get("estimator", "mean"),
        errorbar=barplot_kwargs.get("errorbar", "sd"),
        zorder=barplot_kwargs.get("zorder", min_zorder),
        err_kws=barplot_kwargs.get(
            "err_kws",
            dict(
                color="black",
                zorder=min_zorder + 1,
                linewidth=linewidth * 3,
            ),
        ),
        capsize=barplot_kwargs.get("capsize", 0.2),
        edgecolor=barplot_kwargs.get("edgecolor", "black"),
        linewidth=barplot_kwargs.get("linewidth", linewidth),
        palette=barplot_kwargs.get("palette", palette),
    )
    np.random.seed(stripplot_kwargs.get("seed", 4))
    sns.stripplot(
        data=data,
        x=x,
        y=y,
        ax=ax,
        # hue=x,
        jitter=True,
        zorder=stripplot_kwargs.get("zorder", min_zorder),
        s=stripplot_kwargs.get("s", 4),
        c="xkcd:grey",
        # palette=stripplot_kwargs.get("palette", palette),
        edgecolor=stripplot_kwargs.get("edgecolor", "black"),
        linewidth=stripplot_kwargs.get("linewidth", linewidth),
    )
    groupby = y if orient == "h" else x
    means = data.groupby(groupby, as_index=False).mean()
    sns.scatterplot(
        data=means,
        x=x,
        y=y,
        ax=ax,
        zorder=scatter_kwargs.get("zorder", min_zorder + 2),
        c=scatter_kwargs.get("c", "xkcd:black"),
        s=scatter_kwargs.get("s", 100),
        edgecolor=scatter_kwargs.get("edgecolor", "xkcd:black"),
        linewidth=scatter_kwargs.get("linewidth", linewidth),
    )

    return ax

### Common keyword arguments

In [ ]:
orient = "v"
linewidth = 1
fontdict = dict(size="large")
min_zorder = 3
barplot_kwargs = dict(
    capsize=0.2,
    err_kws=dict(
        color="xkcd:black",
        zorder=min_zorder + 1,
        linewidth=linewidth * 3,
    ),
)
stripplot_kwargs = dict(s=4, seed=5)
scatter_kwargs = dict(s=100, c="xkcd:black", linewidth=linewidth, edgecolor="xkcd:black")
legend_kwargs = dict()

### Raw data

In [ ]:
group_key = "Group"
group1 = "Control"
group2 = "HK1"

variable_args = {
    # ymin, ymax, tick multiple
    "Hemolysis": ((0, 10), 4),
    "MetHb": ((0, 1), 0.4),
    "ROS": ((0, 5e3), 1e3),
    "tBHP-induced ROS": ((0, 5e3), 1e3),
    "diamide-induced ROS": ((0, 5e3), 1e3),
    "PHZ-induced ROS": ((0, 5e4), 1e4),
    "RNS": ((0, 5e4), 1e4),
    "tBHP-induced RNS": ((0, 5e4), 1e4),
    "diamide-induced RNS": ((0, 5e4), 1e4),
    "PHZ-induced RNS": ((0, 5e4), 1e4),
    "Ca": ((0, 2e4), 1e4),
    "Malondialdehyde": ((0, 100), 40),
    "Cytosol: Chymotrypsin-like proteasome activity": ((0, 1.8e5), 6e4),
    "Cytosol: Caspase-like proteasome activity": ((0, 1.8e5), 6e4),
    "Cytosol: Trypsin-like proteasome activity": ((0, 1.8e5), 6e4),
    "Cytosol: Sulfhydryls": ((0, 100), 40),
    "Catalase activity": ((0, 0.10), 0.04),
}

for variable, args in variable_args.items():
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    ax = plot_physiological_data(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            barplot_kwargs=barplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            scatter_kwargs=scatter_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )
    unit = variable_units_dict[variable]
    ylims, tick_muliple = args
    ax.set_title(variable, fontdict=fontdict, y=1.1)
    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.set_ylabel(unit, ha="left", y=1.05, rotation=0)
    ax.set_ylim(ylims)
    ax.grid(which="both")
    ax.yaxis.set_tick_params(length=10, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(tick_muliple))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(tick_muliple, offset=tick_muliple / 2))
    sns.despine(fig)
    output_filepath = output_dirpath / output_filename_fmt.format(
        variable=variable.replace(":", "").strip(), filetype=output_filetype
    )
    fig.tight_layout()
    fig.savefig(output_filepath)
    plt.show()
    plt.close()

### Percentages

In [ ]:
# data_percentages = data / data.groupby("Group").mean().loc["Control"] * 100
# data_percentages["Group"] = data["Group"]

# group_key = "Group"
# group1 = "Control"
# group2 = "HK1"

# variable_args = {
#     # ymin, ymax, tick multiple
#     'Hemolysis': ((0, 400), 50),
#     'MetHb': ((0, 150), 50),
#     'ROS': ((0, 200), 50),
#     'tBHP-induced ROS': ((0, 200), 50),
#     'diamide-induced ROS': ((0, 200), 50),
#     'PHZ-induced ROS': ((0, 200), 50),
#     'RNS': ((0, 200), 50),
#     'tBHP-induced RNS': ((0, 200), 50),
#     'diamide-induced RNS': ((0, 200), 50),
#     'PHZ-induced RNS': ((0, 200), 50),
#     'Ca': ((0, 150), 50),
#     'Malondialdehyde': ((0, 200), 50),
#     'Cytosol: Chymotrypsin-like proteasome activity': ((0, 200), 50),
#     'Cytosol: Caspase-like proteasome activity': ((0, 200), 50),
#     'Cytosol: Trypsin-like proteasome activity': ((0, 200), 50),
#     'Cytosol: Sulfhydryls': ((0, 200), 50),
#     'Catalase activity': ((0, 200), 50),
# }

# for variable, args in variable_args.items():
#     fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
#     ax = plot_physiological_data(
#         data=data_percentages,
#         x=variable if orient == "h" else group_key,
#         y=group_key if orient == "h" else variable,
#         ax=ax,
#         **dict(
#             orient=orient,
#             palette={
#                 group1: "#339966",
#                 group2: "#cc6633",
#             },
#             barplot_kwargs=barplot_kwargs,
#             stripplot_kwargs=stripplot_kwargs,
#             scatter_kwargs=scatter_kwargs,
#             legend_kwargs=legend_kwargs,
#         )
#     )
#     unit = variable_units_dict[variable]
#     ylims, tick_muliple = args
#     ax.set_title(variable, fontdict=fontdict, y=1.1)
#     ax.set_xlabel(group_key, fontdict=fontdict)
#     ax.set_ylabel("%", ha="left", y=1.05, rotation=0)
#     ax.set_ylim(ylims)
#     ax.yaxis.set_tick_params(length=10, which="both")
#     ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(tick_muliple))
#     ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(tick_muliple, offset=tick_muliple/2))
#     sns.despine(fig)
#     output_filepath = output_dirpath / output_filename_fmt.format(variable=variable.replace(":", "").strip() + "_pct", filetype=output_filetype)
#     fig.tight_layout()
#     fig.savefig(output_filepath)
#     plt.close()

# fig